# Using the GPU and mixed precision

_PyTorch (a complete course)_

**Move the model and every batch to the GPU, then run parts in float16 with AMP for a free ~2x speedup.**

Two ideas, stacked. First, put everything on the GPU. A GPU has thousands of small cores that do the same arithmetic on many numbers at once. The matmuls and convolutions at the heart of deep learning are exactly that kind of "do the same thing to a giant grid of numbers" work, so the GPU finishes them far faster than a CPU. Second, use cheaper numbers. Most of that arithmetic does not need full 32-bit precision; 16-bit floats are half the size and run faster on the GPU's specialized units. AMP automatically uses 16-bit where it is safe and 32-bit where it is not.

---

This notebook is a practice scaffold for the **Using the GPU and mixed precision** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ---- 1) THE DEVICE PATTERN: GPU if available, else CPU ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("using device:", device)

# Speed up convolutions when input shapes are fixed (harmless here; key for CNNs).
torch.backends.cudnn.benchmark = True

# ---- 2) A SMALL MODEL + SYNTHETIC DATA, all on the device ----
model = nn.Sequential(
    nn.Linear(1024, 1024), nn.ReLU(),
    nn.Linear(1024, 1024), nn.ReLU(),
    nn.Linear(1024, 10),
).to(device)                       # MOVE THE MODEL (in place; parameters now on 'device')

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()    # expects raw logits + integer class labels

# fake dataset: 8 batches of 256 examples, 1024 features, 10 classes
batches = [(torch.randn(256, 1024), torch.randint(0, 10, (256,))) for _ in range(8)]

# AMP only helps on CUDA; enable it exactly when we are on a GPU.
use_amp = (device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)   # prevents float16 grad underflow

# ---- 3) THE AMP TRAINING STEP ----
model.train()
for epoch in range(3):
    running = 0.0
    for xb, yb in batches:
        xb, yb = xb.to(device), yb.to(device)         # MOVE EVERY BATCH (same device as model!)

        opt.zero_grad()                               # clear last step's grads

        # autocast: run the forward pass in float16 where it is safe
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(xb)                           # logits, shape [256, 10]
            loss = loss_fn(out, yb)

        scaler.scale(loss).backward()                 # scale up, then backprop
        scaler.step(opt)                              # unscale + optimizer.step()
        scaler.update()                               # adjust the scale factor

        running += loss.item()                        # .item(): a plain float, frees the graph
    print(f"epoch {epoch}  avg loss {running / len(batches):.4f}")

# ---- 4) PEAK GPU MEMORY (only meaningful on CUDA) ----
if device.type == 'cuda':
    torch.cuda.synchronize()                          # wait for the GPU before measuring
    mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"peak GPU memory: {mb:.1f} MB")


## Visualize the data & results

_How do training-step time and memory compare for CPU, GPU (FP32), and GPU + AMP?_

In [ ]:
import numpy as np, time

# We MEASURE the CPU FP32 step for real (numpy stands in for the CPU), then derive
# plausible GPU and AMP numbers from typical matmul speedup factors. Labeled illustrative.
rng = np.random.default_rng(0)
B, D = 256, 1024
X  = rng.standard_normal((B, D)).astype(np.float32)
W1 = rng.standard_normal((D, D)).astype(np.float32)
W2 = rng.standard_normal((D, D)).astype(np.float32)

def step():
    h = np.maximum(X @ W1, 0.0)   # linear + ReLU
    return (h @ W2).sum()         # second linear

for _ in range(3): step()          # warmup
N = 20
t0 = time.perf_counter()
for _ in range(N): step()
cpu_ms = (time.perf_counter() - t0) / N * 1000   # measured

gpu_fp32_ms = cpu_ms / 18.0        # GPU ~18x faster on dense matmuls (typical)
gpu_amp_ms  = gpu_fp32_ms / 1.9    # AMP ~1.9x faster than FP32 on tensor cores

print("time ms/step:", [round(cpu_ms, 1), round(gpu_fp32_ms, 1), round(gpu_amp_ms, 1)])
# -> e.g. [169.3, 9.4, 4.9]

# memory: FP32 baseline = 100; AMP keeps float16 activations -> ~0.55x
print("relative memory:", [100, 100, 55])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. The device pattern. Pick device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'), build model = nn.Linear(4, 2), move it with .to(device), and print the device of its first parameter. Predict the output on a CPU-only runtime before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the device once with the is_available() ternary. — _One variable used everywhere prevents CPU/GPU mismatch later._
- Move the model in place with .to(device), then read next(model.parameters()).device. — _Modules move in place; the parameters now live on the chosen device._

**Answer:** import torch
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Linear(4, 2).to(device)
print(device)                                 # cpu  (or cuda on a GPU runtime)
print(next(model.parameters()).device)        # cpu  (or cuda:0)

</details>

**Problem 2.** Type this in Colab. THE #1 GPU bug. Make device = 'cuda' if torch.cuda.is_available() else 'cpu', a model = nn.Linear(4, 2).to(device), and an input x = torch.randn(8, 4) left on the CPU. Run model(x) — on a GPU runtime this raises a device-mismatch error. Then fix it by moving x to the device and print the output shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that x starts on the CPU while the model is on device. — _Every tensor in one operation must share a device, or PyTorch raises Expected all tensors to be on the same device._
- Move the input with x = x.to(device) before the forward pass. — _Plain tensors are not moved in place — you must reassign._

**Answer:** device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = nn.Linear(4, 2).to(device)
x = torch.randn(8, 4)               # still on CPU
# model(x)  # on cuda: RuntimeError: Expected all tensors on the same device
x = x.to(device)                    # the fix: move the input too
print(model(x).shape)               # torch.Size([8, 2])

</details>

**Problem 3.** Type this in Colab. Forgetting .to(device) on a new tensor. With a model and x on device, build a fresh bias b = torch.ones(2) (lands on CPU) and try model(x) + b. Observe the mismatch on a GPU runtime, then create b with device=device and confirm it works.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that tensors you build inside the loop default to the CPU. — _Combining a CPU tensor with a GPU result triggers the same device error._
- Construct it with device=device. — _Create-on-device avoids an extra transfer and the mismatch._

**Answer:** device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = nn.Linear(4, 2).to(device)
x = torch.randn(8, 4, device=device)
# b = torch.ones(2)            # CPU -> model(x) + b errors on a GPU
b = torch.ones(2, device=device)   # fix: build it on the device
print((model(x) + b).shape)        # torch.Size([8, 2])

</details>

**Problem 4.** Type this in Colab. Set up AMP. Create a GradScaler enabled only on CUDA, and write an autocast block for the forward pass using device_type and dtype=torch.float16. Print whether AMP is enabled. (Build use_amp = (device.type == 'cuda').)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- scaler = torch.cuda.amp.GradScaler(enabled=use_amp). — _On CPU there is no float16 speedup, so AMP is disabled cleanly._
- Wrap the forward in with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):. — _autocast runs matmuls in float16 where it is safe, float32 elsewhere._

**Answer:** device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
model = nn.Linear(4, 2).to(device)
x = torch.randn(8, 4, device=device)
with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
    out = model(x)
print("AMP enabled:", use_amp)    # False on CPU, True on a GPU runtime
print(out.shape)                  # torch.Size([8, 2])

</details>

**Problem 5.** Type this in Colab. AMP without a GradScaler (the silent-underflow pitfall) vs the correct order. Write one full AMP step in the RIGHT order: zero_grad &rarr; autocast forward &rarr; scaler.scale(loss).backward() &rarr; scaler.step(opt) &rarr; scaler.update(). Print the loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scale the loss before backward: scaler.scale(loss).backward(). — _Without scaling, tiny float16 gradients underflow to 0 and the model stops learning._
- Step then update: scaler.step(opt) then scaler.update(). — _step unscales and applies the update; update adjusts the scale factor for next time._

**Answer:** torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
model = nn.Linear(4, 2).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
x = torch.randn(8, 4, device=device)
y = torch.randint(0, 2, (8,), device=device)

opt.zero_grad()
with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
    loss = loss_fn(model(x), y)
scaler.scale(loss).backward()     # scale up, then backprop
scaler.step(opt)                  # unscale + optimizer step
scaler.update()                   # adjust scale factor
print(round(loss.item(), 4))      # ~0.70  (near ln(2), random start)

</details>

**Problem 6.** Type this in Colab. Move once, keep on-device. Build batches as CPU tensors, then move each batch inside the loop with .to(device) and bring only the scalar loss back with .item(). Run 3 fake batches and print the average loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Move xb, yb = xb.to(device), yb.to(device) once per batch. — _The model and batch must share a device for the forward pass._
- Accumulate loss.item(), not the tensor. — _.item() copies one scalar back and frees the graph; never .cpu() big tensors in the hot loop._

**Answer:** torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Linear(4, 2).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
batches = [(torch.randn(8, 4), torch.randint(0, 2, (8,))) for _ in range(3)]
running = 0.0
for xb, yb in batches:
    xb, yb = xb.to(device), yb.to(device)   # move every batch
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward(); opt.step()
    running += loss.item()                  # only a scalar comes back
print(round(running / len(batches), 4))     # ~0.7

</details>

**Problem 7.** Type this in Colab. Timing the GPU correctly. On a GPU runtime, time a matmul, calling torch.cuda.synchronize() before reading the clock. Print the elapsed milliseconds. (On CPU the sync call is a harmless no-op — guard it with if device.type == 'cuda'.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call torch.cuda.synchronize() before perf_counter(). — _GPU work is asynchronous; without syncing the CPU clock stops before the GPU finishes._
- Guard the sync so it is skipped on CPU. — _torch.cuda.synchronize() on a CPU-only box would error._

**Answer:** import time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
a = torch.randn(2048, 2048, device=device)
b = torch.randn(2048, 2048, device=device)
if device.type == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
c = a @ b
if device.type == 'cuda': torch.cuda.synchronize()   # wait for the GPU
print(round((time.perf_counter() - t0) * 1000, 2), "ms")   # e.g. ~3 ms on a GPU

</details>

**Problem 8.** Type this in Colab. Peak GPU memory. After running a small AMP training loop, print torch.cuda.max_memory_allocated() in megabytes — but only when on CUDA. Show the CPU-safe guard.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Guard with if device.type == 'cuda': and call torch.cuda.synchronize() first. — _Memory stats are only meaningful on CUDA, and you must wait for pending GPU work._
- Convert max_memory_allocated() bytes to MB by dividing by 1e6. — _It reports the peak allocation across the run._

**Answer:** device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ... after a training loop on the chosen device ...
if device.type == 'cuda':
    torch.cuda.synchronize()
    mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"peak GPU memory: {mb:.1f} MB")
else:
    print("on CPU: no GPU memory stats")   # this prints on a CPU runtime

</details>